# 10. The six bands and the table of site-years

Notebooks 5 to 9 used two of Landsat's bands, red and near infrared. The biomass model of docs/design.md asks whether the decline of clipped biomass is missing from greenness only, or from everything these satellites record. This notebook brings in all six reflective bands, puts the three sensors on one scale band by band, and builds the table that every model is fitted to: one row per site and year, with the clipped biomass and every predictor.

The six bands come from `scripts/extract_landsat.py --all-bands --first-year 1999 --last-year 2020`. Its red, near-infrared and clear-pixel values equal those of the earlier extraction, row for row.

- **Reads** `sites.csv`, `site_year.csv`, `flags.csv`, `era5_land_monthly.csv`, `landsat_conversions.csv` and the files in `landsat_bands/`, all in `data/processed/`, and the herd table of the release.
- **Writes** `landsat_band_conversions.csv` and `model_site_years.csv` to `data/processed/`.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mn_desertification import data, greenness, model, paths, quality
from mn_desertification.greenness import BANDS
from mn_desertification.rules import FLOOR
from mn_desertification.weather import season_weather

site_info = pd.read_csv(paths.PROCESSED_DIR / "sites.csv")

# The six bands at every site and overpass, kept as in step 2: one row a day, at least half of the
# 100 m circle clear, Landsat 7 up to 2020
bands = greenness.clearest_per_day(greenness.add_clear_share(greenness.read_landsat(paths.PROCESSED_DIR / "landsat_bands")))
counted = greenness.counting(bands)
print("Observations that count:", len(counted), "| years:", counted["time"].dt.year.min(), "to", counted["time"].dt.year.max())
print("Without a value:", counted[BANDS].isna().sum().to_dict())

summer = greenness.in_season(counted)   # 1 July to 31 August
summer.groupby("sensor")[BANDS].median().round(3)

Observations that count: 371539 | years: 1999 to 2020
Without a value: {'blue': 22, 'green': 4, 'red': 0, 'nir': 0, 'swir1': 2, 'swir2': 0}


,blue,green,red,nir,swir1,swir2
sensor,,,,,,
landsat5,0.093,0.137,0.161,0.256,0.310,0.250
landsat7,0.089,0.127,0.156,0.257,0.305,0.246
landsat8,0.076,0.120,0.144,0.266,0.301,0.239


The observations are kept by the rules of notebook 6: one row a day, at least half of the 100 m circle clear, and Landsat 7 up to 2020. `swir1` and `swir2` are the two shortwave-infrared bands. A handful of observations have no value in one band, where that band was saturated or out of range in every clear pixel.

## A line per band between the sensors

The method of notebook 7, for six bands: pairs of observations at the same site at most eight days apart, a line fitted by the reduced major axis on the sites with an even number, and the difference that remains on the other sites.

In [2]:
# A line per band between the sensors, fitted as in the sensor checks and tested on sites not used for the fit
measured = greenness.with_indices(summer, to_landsat7=False)
pairs_78 = greenness.sensor_pairs(measured, "landsat7", "landsat8")
pairs_57 = greenness.sensor_pairs(measured, "landsat5", "landsat7")
print("Pairs of Landsat 7 and 8:", len(pairs_78), "| of Landsat 5 and 7:", len(pairs_57))

even = pairs_78["gid"] % 2 == 0
half_fit = greenness.fit_conversion(pairs_78[even], bands=BANDS)
test_rows = []
for band in BANDS:
    odd = pairs_78[~even]
    intercept, slope = half_fit[band]
    as_measured = (odd[band + "_second"] - odd[band + "_first"]).groupby(odd["gid"]).mean().mean()
    fitted = (intercept + slope * odd[band + "_second"] - odd[band + "_first"]).groupby(odd["gid"]).mean().mean()
    test_rows.append({"band": band, "typical value": odd[band + "_first"].median(),
                      "Landsat 8 minus 7, as measured": as_measured, "after the line fitted on other sites": fitted})
pd.DataFrame(test_rows).round(4)

Pairs of Landsat 7 and 8: 42849 | of Landsat 5 and 7: 57538


,band,typical value,"Landsat 8 minus 7, as measured",after the line fitted on other sites
0,blue,0.0898,-0.0113,-0.0012
1,green,0.1292,-0.0052,-0.0009
2,red,0.1577,-0.0091,-0.0007
3,nir,0.2618,0.0041,-0.0006
4,swir1,0.3077,-0.0047,0.0002
5,swir2,0.2456,-0.0041,0.0003


In [3]:
# The lines for use, fitted on every site, and saved
lines = {"landsat5": greenness.fit_conversion(pairs_57, onto="second", bands=BANDS),
         "landsat8": greenness.fit_conversion(pairs_78, bands=BANDS)}
line_table = pd.DataFrame([{"sensor": sensor, "band": band, "intercept": intercept, "slope": slope}
                           for sensor, fitted in lines.items() for band, (intercept, slope) in fitted.items()])
line_table.to_csv(paths.PROCESSED_DIR / "landsat_band_conversions.csv", index=False)

# Red and near infrared must come out as in notebook 7
earlier = pd.read_csv(paths.PROCESSED_DIR / "landsat_conversions.csv")
both = line_table.merge(earlier, on=["sensor", "band"], suffixes=("", "_earlier"))
print("Largest difference from the lines of the sensor checks:",
      float(np.abs(both[["intercept", "slope"]].to_numpy() - both[["intercept_earlier", "slope_earlier"]].to_numpy()).max()))
line_table.pivot(index="band", columns="sensor", values=["intercept", "slope"]).reindex(BANDS).round(4)

Largest difference from the lines of the sensor checks: 1.1102230246251565e-16


intercept             slope         
sensor  landsat5 landsat8 landsat5 landsat8
band                                       
blue     -0.0024   0.0020   0.9794   1.1069
green    -0.0038  -0.0021   0.9542   1.0530
red      -0.0067   0.0047   1.0187   1.0262
nir      -0.0064   0.0004   1.0133   0.9810
swir1    -0.0066  -0.0028   1.0093   1.0243
swir2    -0.0024  -0.0074   1.0017   1.0485

- **As measured, Landsat 8 differs from Landsat 7 in every band,** from −0.011 in blue to +0.004 in the near infrared.
- **After the line, every band is within 0.0012 of Landsat 7** on sites the fit has not seen.
- The lines for use are fitted on every site. Those of red and near infrared equal the lines of notebook 7.

## The predictors and the table

Ten signals, the six bands and four indices, are each summarised four times for every site and year: the median of July and August, of June, of September, and of the July and August of the year before. The weather at the site and the herd of its soum before the summer are joined on, with the clipped biomass.

In [4]:
# The Landsat predictors of every site-year: four seasons of ten signals
converted = greenness.with_indices(counted, conversions=lines)
landsat = model.landsat_predictors(converted)
print("Site-years:", len(landsat), "| columns:", landsat.shape[1])
seasons = {"July and August": "ndvi", "June": "ndvi_june", "September": "ndvi_september", "the summer before": "ndvi_before"}
in_decade = landsat[landsat["year"].between(2011, 2020)]
pd.Series({season: in_decade[column].notna().sum() for season, column in seasons.items()}, name="site-years with a value, 2011-2020").to_frame().T

Site-years: 34196 | columns: 43


,July and August,June,September,the summer before
"site-years with a value, 2011-2020",14782,13933,14550,14782


In [5]:
# The table of site-years: the target and every predictor
flags = pd.read_csv(paths.PROCESSED_DIR / "flags.csv")
site_year = pd.read_csv(paths.PROCESSED_DIR / "site_year.csv")
biomass = quality.drop_flagged(site_year, flags)[["gid", "year", "bms"]].copy()
biomass["log_bms"] = np.log10(biomass["bms"].clip(lower=FLOOR))

weather = season_weather(pd.read_csv(paths.PROCESSED_DIR / "era5_land_monthly.csv"), "all")
weather["log_rain"] = np.log10(weather["rain_growing"])

herd = data.read_herd_before_summer().stack().rename("herd").reset_index()
herd.columns = ["asid", "year", "herd"]
herd["log_herd"] = np.log10(herd["herd"])

table = (biomass.merge(site_info[["gid", "asid", "zone"]], on="gid")
         .merge(weather[["gid", "year", "log_rain", "temp_summer"]], on=["gid", "year"], how="left")
         .merge(landsat, on=["gid", "year"], how="left")
         .merge(herd[["asid", "year", "log_herd"]], on=["asid", "year"], how="left"))
table["aimag"] = table["asid"] // 100
table.to_csv(paths.PROCESSED_DIR / "model_site_years.csv", index=False)

print("Site-years:", len(table), "| sites:", table["gid"].nunique(), "| years:", table["year"].min(), "to", table["year"].max(),
      "| columns:", table.shape[1])
decade = table[table["year"].between(2011, 2020)]
pd.DataFrame({rung: {"predictors": len(names), "site-years with all of them": int(decade[names].notna().all(axis=1).sum()),
                     "values missing": f"{decade[names].isna().mean().mean():.1%}"}
              for rung, names in model.RUNGS.items()}).T

Site-years: 18149 | sites: 1488 | years: 2007 to 2020 | columns: 51


,predictors,site-years with all of them,values missing
1 weather,2,13367,0.0%
2 greenness,1,13282,0.6%
3 all bands,9,13282,0.6%
4 the season,30,12183,3.1%
5 herds,1,13334,0.2%


The table holds 13,367 site-years in 2011–2020, the same as in notebook 2, and little is missing: 0.6% of the greenness and band values, and 3.1% of those of the season.

## A first look

How each July and August signal goes with biomass. Between sites compares the sites' ten-year averages. Within sites compares each site's years with its own average, which is what the models have to explain.

In [6]:
# A first look: how each July and August signal goes with biomass, between sites and within sites
signals = BANDS + ["ndvi", "msavi", "nir_swir1", "swir1_swir2"]
columns = ["log_bms", *signals]
site_means = decade.groupby("gid")[columns].mean()
departures = decade[columns] - decade.groupby("gid")[columns].transform("mean")
pd.DataFrame({"between sites": site_means.corr()["log_bms"], "within sites": departures.corr()["log_bms"]}).loc[signals].round(2).T

,blue,green,red,nir,swir1,swir2,ndvi,msavi,nir_swir1,swir1_swir2
between sites,-0.71,-0.69,-0.68,0.09,-0.32,-0.57,0.75,0.72,0.51,0.70
within sites,-0.34,-0.35,-0.39,0.12,-0.36,-0.40,0.38,0.36,0.35,0.36


- **Between sites, greener and darker in the visible bands means more biomass.** NDVI correlates at 0.75, and blue, green and red at about −0.7.
- **Within sites every link is weaker,** at 0.34 to 0.40 in size, apart from the near infrared.
- **No signal stands out.** The second shortwave-infrared band and red do as well as NDVI.